# Process Items JSONL

Reads raw gzipped JSONL metadata files (`meta_<category>.jsonl.gz`), filters to only items that appear in `reviews.csv`, and writes a clean intermediate CSV for `prepare-dataset.ipynb`.

In [5]:
import json
import gzip
from pathlib import Path
from typing import Any
import pandas as pd

In [6]:
# --- Config ---
CATEGORIES = ["Beauty_and_Personal_Care", "Clothing_Shoes_and_Jewelry"]
FIELDS_TO_KEEP: list[str] = ["parent_asin", "title", "price", "store"]
DATA_DIR: str = "../data"
REVIEWS_FILE: str = "reviews.csv"
OUTPUT_FILE: str = "items.csv"

## Common utils

In [7]:
def stream_jsonl(path: str, fields: list[str] | None = None):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for _, line in enumerate(f):
            obj = json.loads(line)
            if fields is not None:
                obj = {k: obj.get(k) for k in fields}
            yield obj

## Load valid parent_asins from reviews

In [8]:
reviews_path = Path(DATA_DIR) / REVIEWS_FILE
reviews_df = pd.read_csv(reviews_path, usecols=["parent_asin"])
valid_asins: set[str] = set(reviews_df["parent_asin"].unique())
print(f"Loaded {len(valid_asins):,} unique parent_asins from {REVIEWS_FILE}")

Loaded 125,873 unique parent_asins from reviews.csv


## Stream and filter items from meta files

In [9]:
items: list[dict[str, Any]] = []
for cat in CATEGORIES:
    path = f"{DATA_DIR}/meta_{cat}.jsonl.gz"
    print(f"Loading items: {path}")
    for obj in stream_jsonl(path, fields=FIELDS_TO_KEEP):
        asin = obj.get("parent_asin")
        if asin is None or asin not in valid_asins:
            continue
        obj["category"] = cat
        items.append(obj)

print(f"Loaded {len(items):,} items")

Loading items: ../data/meta_Beauty_and_Personal_Care.jsonl.gz
Loading items: ../data/meta_Clothing_Shoes_and_Jewelry.jsonl.gz
Loaded 125,873 items


In [10]:
df = pd.DataFrame(items)
display(df.head())
df.info()

,parent_asin,title,price,store,category
0,B01DX1OEFO,"L.A. COLORS 5 Color Matte Eyeshadow, Brown Twe...",2.49,L.A. COLORS,Beauty_and_Personal_Care
1,B0BVYFF3ML,Ayana Marley Hair 6 Packs Marley Twist Braidin...,34.99,Ayana,Beauty_and_Personal_Care
2,B09WHMND5W,Newcally Natural Look Lashes Mink 3D Fluffy Wi...,9.45,Newcally,Beauty_and_Personal_Care
3,B01M67WKPZ,BLUE LIZARD Australian Sunscreen Blue Lizard S...,27.8,BLUE LIZARD,Beauty_and_Personal_Care
4,B07PN759Z4,Shea Butter Clean Hand Wash by South of France...,22.69,South of France Natural Body Care,Beauty_and_Personal_Care


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125873 entries, 0 to 125872
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   parent_asin  125873 non-null  object
 1   title        125873 non-null  object
 2   price        83793 non-null   object
 3   store        122667 non-null  object
 4   category     125873 non-null  object
dtypes: object(5)
memory usage: 4.8+ MB


## Export to CSV

In [11]:
output_path = Path(DATA_DIR)
output_path.mkdir(parents=True, exist_ok=True)
file_path = output_path / OUTPUT_FILE
df.to_csv(file_path, index=False)
print(f"Wrote {file_path} ({df.shape[0]:,} rows, {df.shape[1]} columns)")

Wrote ../data/items.csv (125,873 rows, 5 columns)
